In [ ]:
!which python

/home/sa1/Documents/gitReps/pythonQuickAndDirty/venv/bin/python


In [ ]:
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src" / "data" / "SmartEM" / "Philips").exists():
    repo_root = repo_root.parents[2]

data_dir = repo_root / "src" / "data" / "SmartEM" / "Philips"

input_data = pd.read_csv(data_dir / "input.csv")
output_data = pd.read_csv(data_dir / "output.csv")

input_data, output_data

In [ ]:
import re

odd_columns = ["input_6", "input_8", "input_9"]
expected_dims = {"input_6": 2, "input_8": 4, "input_9": 3}
token_pattern = re.compile(r"^\d+(?:-\d+)*$")

input_processed = input_data.copy()


def parse_tokens(value):
    text = str(value).strip()
    if not token_pattern.fullmatch(text):
        raise ValueError(f"Unexpected token format in odd column: {value!r}")
    return [int(part) for part in text.split("-")]


def normalize_tokens(tokens, width):
    if len(tokens) > width:
        raise ValueError(f"Token length {len(tokens)} exceeds expected width {width}")
    padded = tokens + [0] * (width - len(tokens))
    return "-".join(str(v) for v in padded)


observed_dims = {}
for col in odd_columns:
    parsed = input_data[col].astype("string").map(parse_tokens)
    observed_dims[col] = int(parsed.map(len).max())
    input_processed[col] = parsed.map(lambda t: normalize_tokens(t, expected_dims[col]))

assert observed_dims == expected_dims, f"Unexpected max dims: {observed_dims}"

# Spot-check requested examples.
assert normalize_tokens(parse_tokens("189"), expected_dims["input_6"]) == "189-0"
assert normalize_tokens(parse_tokens("340-260"), expected_dims["input_8"]) == "340-260-0-0"
assert normalize_tokens(parse_tokens("651-360-455-312"), expected_dims["input_8"]) == "651-360-455-312"
assert normalize_tokens(parse_tokens("0"), expected_dims["input_8"]) == "0-0-0-0"

# Keep dataset unchanged except offending columns.
untouched_columns = [c for c in input_data.columns if c not in odd_columns]
for col in untouched_columns:
    assert input_processed[col].equals(input_data[col]), f"Unexpected change in {col}"

assert list(input_processed.columns) == list(input_data.columns)
assert len(input_processed) == len(input_data)

# Verify fixed widths in-place.
for col, width in expected_dims.items():
    lens = input_processed[col].astype("string").str.split("-").str.len()
    assert lens.eq(width).all(), f"Width mismatch in {col}"

processing_summary = {
    "shape": input_processed.shape,
    "column_count": len(input_processed.columns),
    "changed_columns": odd_columns,
    "observed_dims": observed_dims,
    "samples": {
        col: input_processed[col].head(3).tolist() for col in odd_columns
    },
}

processing_summary



In [ ]:
input_processed_path = data_dir / "input_processed.csv"
input_processed.to_csv(input_processed_path, index=False)

{
    "saved_to": str(input_processed_path),
    "shape": input_processed.shape,
    "column_count": len(input_processed.columns),
    "changed_columns": odd_columns,
}

